# Esquema: clasificación multiclase + sklearn + PyTorch (MVP)

CSV → target **0..K-1** → features → split → modelos **sklearn** → red **PyTorch** → comparación en test.

| Paso | Contenido |
|------|-----------|
| 1–5 | CSV (`data/datos_flores.csv`), codificación del target, features, split |
| 6 | Modelos **sklearn** (`build_models(N_CLASSES)`) |
| 7 | Red **PyTorch** (`TabularMultiNet`) |
| 8 | **Comparación global** + reporte del mejor |

> Ejecuta el notebook desde `13-esquemas-sklearn-pytorch/`.


## 1. CSV

In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_flores.csv")


## 2. Target multiclase

In [36]:
df = df.dropna(subset=["especie"]).copy()
ORDEN_CLASES = ["setosa", "versicolor", "virginica"]
MAPA_MULTI = {n: i for i, n in enumerate(ORDEN_CLASES)}
y = df["especie"].str.strip().str.lower().map(MAPA_MULTI).astype(int)


## 3. Features

In [37]:
cols_num = ["sepal_length", "sepal_width"]
X = df[cols_num].astype(float).copy()
for col in cols_num:
    X[col] = X[col].fillna(X[col].median())


## 4. Split

In [38]:
def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split)."""
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test


RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=True
)
print(f"Tamaños → train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")


Tamaños → train: 17 | val: 6 | test: 6


## 6. Entrenar modelos sklearn

**Fit** solo en train; métricas en el paso 8.

Cada `Pipeline` define **por separado**:
- **`transformacion`**: `SimpleImputer` (imputación de faltantes; complementa pandas pasos 3–4).
- **`estandarizado`**: `StandardScaler` (media 0, desv. 1).
- **`modelo`**: estimador de `build_models()`.


In [39]:
def build_models(n_classes):
    """Comenta entradas del dict para excluir modelos."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    if n_classes < 3:
        raise ValueError(f"build_models: K={n_classes} < 3 (multiclase)")
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            objective="multi:softmax",
            num_class=n_classes,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
            loss_function="MultiClass",
        ),
    }


RANDOM_STATE = 42
N_CLASSES = int(y.nunique())
MODELS = build_models(N_CLASSES)

def build_sklearn_pipeline(modelo):
    """Pipeline con transformación y estandarizado como pasos separados."""
    return Pipeline([
        ("transformacion", SimpleImputer(strategy="median")),  # imputación (transformación)
        ("estandarizado", StandardScaler()),                   # media 0, desv. 1
        ("modelo", modelo),
    ])


pipelines = {}
for nombre, modelo in MODELS.items():
    pipe = build_sklearn_pipeline(modelo)
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe



## 7. Red neuronal (PyTorch)

Misma arquitectura tabular. Salida con **K logits** + `CrossEntropyLoss`.


In [40]:
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# Transformación (imputación; stats solo de train)
imputer_nn = SimpleImputer(strategy="median")
X_tr = imputer_nn.fit_transform(X_train)
X_va = imputer_nn.transform(X_val)
X_te = imputer_nn.transform(X_test)

# Estandarizado (media/std solo de train)
scaler_nn = StandardScaler()
X_tr = scaler_nn.fit_transform(X_tr)
X_va = scaler_nn.transform(X_va)
X_te = scaler_nn.transform(X_te)
n_in = X_tr.shape[1]
n_classes = len(ORDEN_CLASES)


class TabularMultiNet(nn.Module):
    """MLP tabular; salida K logits (multiclase)."""

    def __init__(self, n_features: int, n_classes: int, dropout_rate: float = 0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)  # (batch, n_classes)


modelo_nn = TabularMultiNet(n_in, n_classes).to(device)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=1e-2)

X_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_t = torch.tensor(y_train.values, dtype=torch.long, device=device)

for _ in range(400):
    modelo_nn.train()
    optimizador.zero_grad()
    criterio(modelo_nn(X_t), y_t).backward()
    optimizador.step()



## 8. Análisis comparativo (sklearn + PyTorch)

Elige el mejor modelo por **accuracy en val**; reporte en **test** del ganador.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


In [41]:
filas = []
predicciones_test = {}
for nombre, pipe in pipelines.items():
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)
    acc_val = accuracy_score(y_val, pred_val)
    acc_test = accuracy_score(y_test, pred_test)
    predicciones_test[nombre] = pred_test
    filas.append(
        {"modelo": nombre, "accuracy_val": acc_val, "accuracy_test": acc_test}
    )

modelo_nn.eval()
with torch.no_grad():
    pred_nn_val = modelo_nn(torch.tensor(X_va, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    pred_nn = modelo_nn(torch.tensor(X_te, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()

acc_nn_val = accuracy_score(y_val, pred_nn_val)
acc_nn_test = accuracy_score(y_test, pred_nn)
predicciones_test["PyTorch_MLP"] = pred_nn

comparacion = pd.concat(
    [
        pd.DataFrame(filas),
        pd.DataFrame(
            [{"modelo": "PyTorch_MLP", "accuracy_val": acc_nn_val, "accuracy_test": acc_nn_test}]
        ),
    ],
    ignore_index=True,
).sort_values("accuracy_val", ascending=False)

display(comparacion.round(4))

mejor_nombre = comparacion.iloc[0]["modelo"]
mejor = comparacion.iloc[0]
print(f"\nMejor modelo en val: {mejor_nombre} (accuracy_val = {mejor['accuracy_val']:.4f})")
print(f"Accuracy en test del ganador: {mejor['accuracy_test']:.4f}")
print(
    classification_report(
        y_test,
        predicciones_test[mejor_nombre],
        target_names=ORDEN_CLASES,
    )
)



,modelo,accuracy_val,accuracy_test
7,RandomForest,0.8333,0.3333
10,XGBoost,0.6667,0.5000
8,GradientBoosting,0.6667,0.3333
6,DecisionTree,0.6667,0.3333
1,SGDClassifier,0.6667,0.6667
11,CatBoost,0.6667,0.5000
3,OneVsOneClassifier,0.5000,0.5000
2,SVC,0.5000,0.5000
0,LogisticRegression,0.5000,0.5000
4,OneVsRestClassifier,0.5000,0.5000



Mejor modelo en val: RandomForest (accuracy_val = 0.8333)
Accuracy en test del ganador: 0.3333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         2
  versicolor       0.00      0.00      0.00         2
   virginica       0.00      0.00      0.00         2

    accuracy                           0.33         6
   macro avg       0.33      0.33      0.33         6
weighted avg       0.33      0.33      0.33         6

